# Getting financial data from APIs with Python

A hands-on walkthrough of each free data source in the portfolio plan, one section per source. Every section follows the same steps:

1. **What you get** and what it costs
2. **Access**: how to get a key and where to put it
3. **The request**: URL, parameters and authentication
4. **Code**: plain `requests` calls you can read line by line
5. **Parsing**: turning the response into a Polars DataFrame
6. **Limits and gotchas**

The code is deliberately plain so you can see how each API works. In your project, call the matching function in `finance_data.py` instead: it sends the same requests and adds caching, retries and rate limiting.

**How to run it:** run the setup cells first, then any section from top to bottom. Sections don't depend on each other. If a section needs a key you haven't set, it stops with a message naming the key.

| § | Source | Key | Authentication | Free limit | Best for |
|---|---|---|---|---|---|
| 1 | SEC EDGAR | none | `User-Agent` header with your name and email | 10 requests/second | Filings, reported financials |
| 2 | FRED / ALFRED | free | `api_key` parameter | about 120 requests/minute | Rates, macro data, vintages |
| 3 | Tiingo | free | `Authorization: Token …` header | 50/hour, 1,000/day, 500 symbols/month | Daily prices, raw and adjusted |
| 4 | Alpaca | free | key ID and secret headers | 200/minute | Daily and intraday bars, many symbols per call |
| 5 | Alpha Vantage | free | `apikey` parameter | 25/day | Fundamentals, earnings dates, delisted symbols |
| 6 | Databento | $125 credit | official Python client | pay per byte | Tick and order-book data |
| 7 | Stooq | free (captcha) | `apikey` parameter | daily download cap | Global daily prices |
| 8 | yfinance | none | none (reads Yahoo's website) | unofficial, throttled | Quick prototypes only |
| A | SEC insider data, Ken French factors | none | plain file download | none | Insider trades, factor returns |

## How an API request works

Every section is a variation on this pattern:

```python
resp = requests.get(
    "https://api.example.com/v1/prices",               # base URL + endpoint
    params={"symbol": "AAPL", "start": "2024-01-01"},  # query parameters, sent after "?"
    headers={"Authorization": "Token abc123"},         # authentication, client identity
    timeout=30,                                        # never wait forever
)
resp.raise_for_status()  # turn 4xx/5xx status codes into a Python exception
data = resp.json()       # parse the JSON body into dicts and lists
```

| Status | Meaning |
|---|---|
| `200` | OK (but Alpha Vantage and Stooq also send errors with 200: see §5 and §7) |
| `400`, `422` | a parameter is wrong |
| `401`, `403` | missing or wrong key, or your plan doesn't include that data |
| `404` | wrong URL or unknown symbol |
| `429` | too many requests: slow down |
| `5xx` | server problem: retry later |

When a call fails, print `resp.text`: most APIs explain the problem in the body.

## 0. Setup

In a terminal, in your project folder:

```bash
uv add polars pandas requests python-dotenv
uv add alpaca-py databento yfinance   # only needed for §4b, §6 and §8
uv add --dev ipykernel                # lets VS Code run this notebook with the project's .venv
```

Copy `.env.example` to `.env` and fill in the keys you have. `python-dotenv` loads that file into environment variables, so keys never appear in your code or your Git history. Before your first commit, add these lines to `.gitignore`:

```
.env
data/
```

Tables in this notebook are Polars DataFrames. Call `.to_pandas()` whenever a library such as statsmodels or seaborn wants pandas.

In [ ]:
import io
import os
import time
import zipfile
from datetime import date, datetime, timedelta
from pathlib import Path

import polars as pl
import requests
from dotenv import load_dotenv

load_dotenv(override=True)  # read .env; re-run this cell after editing .env

DATA_DIR = Path("data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def require(name: str) -> str:
    """Return an environment variable, or stop with a message naming it."""
    value = os.getenv(name, "").strip()
    if not value:
        raise RuntimeError(f"{name} is not set. Add it to .env, then re-run the setup cell.")
    return value


for name in ["SEC_USER_AGENT", "FRED_API_KEY", "TIINGO_API_KEY", "ALPACA_API_KEY", "ALPACA_SECRET_KEY",
             "ALPHAVANTAGE_API_KEY", "DATABENTO_API_KEY", "STOOQ_API_KEY"]:
    print(f"{name:<22} {'set' if os.getenv(name) else 'missing'}")

## 1. SEC EDGAR: filings and reported financials

**What you get:** every filing a US-listed company makes (10-K, 10-Q, 8-K, Form 4 …) and every number tagged in its financial statements (XBRL data, from 2009 for the largest companies and 2011 for the rest). Free, with no key and no account.

**Access rule:** the SEC's fair-access policy requires a `User-Agent` header with your name and an email address, for example `Jane Doe jane.doe@gmail.com`. Without it you get `403 Forbidden`, and your IP address may be blocked for a while. Stay under **10 requests per second**.

| Endpoint | Returns |
|---|---|
| `https://www.sec.gov/files/company_tickers.json` | ticker → CIK map for current filers |
| `https://data.sec.gov/submissions/CIK##########.json` | company details and filing history |
| `https://data.sec.gov/api/xbrl/companyconcept/CIK##########/us-gaap/{Concept}.json` | one line item, every period, one company |
| `https://data.sec.gov/api/xbrl/companyfacts/CIK##########.json` | every line item, one company |
| `https://data.sec.gov/api/xbrl/frames/us-gaap/{Concept}/{Unit}/{Period}.json` | one line item, one period, every company |

`##########` is the company's **CIK** (Central Index Key) padded with zeros to 10 digits: Apple's CIK 320193 becomes `CIK0000320193`. An unpadded CIK returns 404.

In [ ]:
SEC_HEADERS = {
    "User-Agent": require("SEC_USER_AGENT"),  # "Your Name your.email@example.com"
    "Accept-Encoding": "gzip, deflate",
}


def sec_get(url: str) -> dict:
    """GET a SEC JSON endpoint politely and return the parsed JSON."""
    resp = requests.get(url, headers=SEC_HEADERS, timeout=30)
    if resp.status_code == 403:
        raise RuntimeError("SEC returned 403: check SEC_USER_AGENT (name + email) and your request rate.")
    resp.raise_for_status()
    time.sleep(0.15)  # stays well under 10 requests per second
    return resp.json()

### Step 1: ticker → CIK

The SEC identifies companies by CIK, not by ticker. `company_tickers.json` maps current tickers to CIKs. It lists only companies that file today; for a delisted company, look up the CIK with EDGAR's company search.

In [ ]:
tickers_json = sec_get("https://www.sec.gov/files/company_tickers.json")
# {"0": {"cik_str": 320193, "ticker": "AAPL", "title": "Apple Inc."}, "1": {...}, ...}

tickers = pl.DataFrame(list(tickers_json.values())).rename({"cik_str": "cik"})
print(f"{tickers.height:,} tickers")

cik = tickers.filter(pl.col("ticker") == "AAPL").item(0, "cik")
cik10 = f"{cik:010d}"  # 320193 -> "0000320193"
print(cik, cik10)

### Step 2: filing history (`submissions`)

The response holds company details plus `filings.recent`: parallel lists with one entry per filing, covering the last year or the latest 1,000 filings, whichever is more. Older filings sit in extra files listed under `filings.files`, fetched from `https://data.sec.gov/submissions/<name>`.

In [ ]:
sub = sec_get(f"https://data.sec.gov/submissions/CIK{cik10}.json")
print(sub["name"], "|", sub["sicDescription"], "| fiscal year ends (MMDD):", sub["fiscalYearEnd"])

recent = sub["filings"]["recent"]  # a dict of equal-length lists
filings = pl.DataFrame({
    "form": recent["form"],
    "filing_date": recent["filingDate"],
    "report_date": recent["reportDate"],
    "accepted": recent["acceptanceDateTime"],
    "accession": recent["accessionNumber"],
    "items": recent["items"],
    "primary_doc": recent["primaryDocument"],
}).with_columns(
    pl.col("filing_date").str.to_date(),
    pl.col("report_date").str.to_date(strict=False),  # "" for filings without a period -> null
    pl.col("accepted").str.slice(0, 19).str.to_datetime("%Y-%m-%dT%H:%M:%S"),
)
filings.filter(pl.col("form").is_in(["10-K", "10-Q"])).head()

`accepted` (EDGAR's `acceptanceDateTime`) ends in `Z`, which normally means UTC. Before using it to decide whether a filing came out before or after the market close, compare a few values with the acceptance time shown on the filing's EDGAR index page.

8-K filings whose `items` include **2.02** (Results of Operations and Financial Condition) are earnings releases: ready-made event dates for an earnings study. The accession number also gives you the link to any filing's documents:

In [ ]:
earnings_8k = filings.filter((pl.col("form") == "8-K") & pl.col("items").str.contains("2.02", literal=True))
print(f"{earnings_8k.height} earnings releases (8-K item 2.02) among the recent filings")

latest_10k = filings.filter(pl.col("form") == "10-K").row(0, named=True)  # filings are newest first
folder = latest_10k["accession"].replace("-", "")
print(f"https://www.sec.gov/Archives/edgar/data/{cik}/{folder}/{latest_10k['primary_doc']}")

### Step 3: one line item over time (`companyconcept`)

Line items are US-GAAP concept names such as `NetIncomeLoss`, `Assets`, `StockholdersEquity`, `EarningsPerShareDiluted` or `RevenueFromContractWithCustomerExcludingAssessedTax`. Companies choose among similar tags, so check which ones yours uses (Step 4 lists them all).

Shape of the response (values shortened):

```json
{"cik": 320193, "taxonomy": "us-gaap", "tag": "NetIncomeLoss", "label": "…",
 "units": {"USD": [{"start": "…", "end": "…", "val": …, "fy": …, "fp": "FY",
                    "form": "10-K", "filed": "…", "accn": "…", "frame": "CY2024"}, …]}}
```

- `start`, `end`: the period the number covers (balance-sheet items have only `end`)
- `filed`: the day the number became public
- `form`, `accn`: the filing it came from
- `frame`: set on the one value the SEC picked for each calendar period

In [ ]:
concept = sec_get(f"https://data.sec.gov/api/xbrl/companyconcept/CIK{cik10}/us-gaap/NetIncomeLoss.json")

net_income = pl.DataFrame(concept["units"]["USD"], infer_schema_length=None).with_columns(
    pl.col("start", "end", "filed").str.to_date()
)
print(f"{net_income.height} values (a period repeats when later filings show it as a comparative)")

annual = net_income.filter(pl.col("frame").str.contains(r"^CY\d{4}$"))  # one value per calendar year
annual.select("frame", "start", "end", "val", "form", "filed").tail()

**Point-in-time rule:** a backtest standing on day D may only use rows with `filed < D`. Filtering on `end` instead uses numbers that weren't public yet: a 10-K arrives two to three months after the fiscal year ends.

`infer_schema_length=None` tells Polars to look at every row before choosing column types. API responses often leave out a field in early rows or mix whole numbers and decimals, so use it whenever you build a DataFrame from a list of records.

### Step 4: every line item for one company (`companyfacts`)

The same records for every concept, nested as `facts → taxonomy → concept → units → unit → [records]`. Flatten it into one long table:

In [ ]:
facts = sec_get(f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json")

rows = [
    {"taxonomy": taxonomy, "concept": concept_name, "unit": unit, **record}
    for taxonomy, concepts in facts["facts"].items()     # "dei", "us-gaap"
    for concept_name, body in concepts.items()           # "Revenues", "Assets", ...
    for unit, records in body["units"].items()           # "USD", "shares", "USD/shares"
    for record in records
]
all_facts = pl.DataFrame(rows, infer_schema_length=None).with_columns(
    pl.col("start", "end", "filed").str.to_date(),
    pl.col("val").cast(pl.Float64),
)
print(f"{all_facts.height:,} facts, {all_facts['concept'].n_unique()} concepts")

# Which revenue tags does this company use, and over which years?
(
    all_facts.filter(pl.col("concept").str.contains("Revenue"))
    .group_by("concept")
    .agg(pl.len().alias("facts"), pl.col("end").min().alias("first"), pl.col("end").max().alias("last"))
    .sort("facts", descending=True)
)

### Step 5: one number for every company (`frames`)

Periods: `CY2024` for a calendar year (flows such as revenue), `CY2024Q4` for a quarter, `CY2024Q4I` for an instant (balance-sheet items such as assets). Units: `USD`, `shares`, and `USD-per-shares` for per-share values such as EPS. For each company the SEC picks the value that best fits the calendar period, so companies with different fiscal years can be compared.

In [ ]:
frame = sec_get("https://data.sec.gov/api/xbrl/frames/us-gaap/Assets/USD/CY2024Q4I.json")
# {"tag": "Assets", "ccp": "CY2024Q4I", "uom": "USD", "pts": <number of companies>, "data": [...]}

assets = pl.DataFrame(frame["data"]).with_columns(pl.col("end").str.to_date())
print(f"{frame['pts']:,} companies reported Assets for {frame['ccp']}")
assets.sort("val", descending=True).head(10)

### Limits and gotchas

- **403**: missing or generic `User-Agent`, or more than 10 requests per second.
- **404**: CIK not padded to 10 digits, or the company never used that concept.
- **Duplicates**: a period reappears in later filings as a comparative, and amendments restate it. For backtests keep the first-filed value; for the latest restated number keep the last.
- **Many companies**: rather than thousands of API calls, download the bulk files [`companyfacts.zip`](https://www.sec.gov/Archives/edgar/daily-index/xbrl/companyfacts.zip) and [`submissions.zip`](https://www.sec.gov/Archives/edgar/daily-index/bulkdata/submissions.zip). They're large, so fetch them once.

**In your project:** `from finance_data import sec_cik, sec_filings, sec_company_facts, sec_frames`

## 2. FRED and ALFRED: economic and market time series

**What you get:** more than 800,000 US and international series from the St. Louis Fed: Treasury yields, the fed funds rate, CPI, unemployment, GDP, credit spreads, the VIX, recession dates. **ALFRED** keeps every past version (vintage) of each series.

**Access:** create an account at [fredaccount.stlouisfed.org](https://fredaccount.stlouisfed.org), open *API Keys* and request a key (a sentence about your project is enough). The key is a 32-character lowercase string. Save it as `FRED_API_KEY`.

**The request:** `GET https://api.stlouisfed.org/fred/series/observations`

| Parameter | Meaning | Example |
|---|---|---|
| `series_id` | the series (required) | `DGS10` |
| `api_key` | your key (required) | |
| `file_type` | response format; **the default is XML**, so always send `json` | `json` |
| `observation_start`, `observation_end` | date range | `2000-01-01` |
| `units` | transformation: `lin` level, `chg` change, `pch` % change, `pc1` % change from a year ago, `log` | `pc1` |
| `frequency`, `aggregation_method` | convert to a lower frequency (`w`, `m`, `q`, `a`) using `avg`, `sum` or `eop` (end of period) | `m`, `eop` |
| `realtime_start`, `realtime_end` | ALFRED: the data as it was published on past dates | `2020-01-01` |
| `limit`, `offset` | paging: at most 100,000 rows per request | |

Useful series: `DGS3MO`, `DGS2`, `DGS10` (Treasury yields), `T10Y3M` (10-year minus 3-month), `FEDFUNDS`, `CPIAUCSL` (CPI), `UNRATE`, `PAYEMS` (payrolls), `USREC` (recession indicator), `VIXCLS`, `BAA10Y` (Baa corporate bonds minus 10-year Treasury).

In [ ]:
FRED_KEY = require("FRED_API_KEY")


def fred_get(endpoint: str, **params) -> dict:
    """Call https://api.stlouisfed.org/fred/<endpoint> and return the parsed JSON."""
    params.update(api_key=FRED_KEY, file_type="json")
    resp = requests.get(f"https://api.stlouisfed.org/fred/{endpoint}", params=params, timeout=30)
    if resp.status_code != 200:
        # FRED explains errors in the body: {"error_code": 400, "error_message": "..."}
        raise RuntimeError(f"FRED {resp.status_code}: {resp.text[:300]}")
    return resp.json()

### Find a series ID

`series/search` looks through titles, units and tags. The results come back under the key `seriess`, with two s's: that's FRED's own spelling.

In [ ]:
found = fred_get("series/search", search_text="treasury constant maturity 10-year",
                 order_by="popularity", sort_order="desc", limit=5)
pl.DataFrame(found["seriess"]).select(
    "id", "title", "frequency_short", "units_short", "observation_start", "observation_end", "popularity"
)

### Get the observations

Values arrive as strings, and `"."` marks a day without a value (a market holiday), so cast with `strict=False` to turn those into nulls.

In [ ]:
obs = fred_get("series/observations", series_id="DGS10", observation_start="2000-01-01")
print(obs["count"], "observations; units:", obs["units"])
# obs["observations"] = [{"realtime_start": ..., "realtime_end": ..., "date": "2000-01-03", "value": "6.58"}, ...]

dgs10 = pl.DataFrame(obs["observations"]).select(
    pl.col("date").str.to_date(),
    pl.col("value").cast(pl.Float64, strict=False).alias("DGS10"),  # "." -> null
)
dgs10.tail()

### Several series in one table

A helper that returns one column named after the series, then full joins on `date`:

In [ ]:
def fred_column(series_id: str, **params) -> pl.DataFrame:
    """One FRED series as a (date, <series_id>) table."""
    observations = fred_get("series/observations", series_id=series_id, **params)["observations"]
    return pl.DataFrame(observations).select(
        pl.col("date").str.to_date(),
        pl.col("value").cast(pl.Float64, strict=False).alias(series_id),
    )


curve = fred_column("DGS3MO", observation_start="2000-01-01")
for series_id in ["DGS2", "DGS10"]:
    curve = curve.join(fred_column(series_id, observation_start="2000-01-01"), on="date", how="full", coalesce=True)

curve = curve.sort("date").with_columns((pl.col("DGS10") - pl.col("DGS3MO")).alias("spread_10y_3m"))
curve.tail()

### Let FRED do the math

`units` and `frequency` are applied on FRED's side, so you download less and compute less:

In [ ]:
inflation = fred_column("CPIAUCSL", units="pc1", observation_start="2015-01-01")  # % change from a year ago
print(inflation.tail(3))

dgs10_month_end = fred_column("DGS10", frequency="m", aggregation_method="eop", observation_start="2015-01-01")
print(dgs10_month_end.tail(3))

### ALFRED: the data as it was known at the time

Macro numbers get revised. A month's payrolls figure is published, revised in each of the next two months and again in the yearly benchmark revision, often by tens of thousands of jobs. A strategy tested on today's revised numbers uses information nobody had at the time.

Ask for the whole real-time range and FRED returns one row per published version, with the window during which that version was the official number:

In [ ]:
vintages = fred_get(
    "series/observations",
    series_id="PAYEMS",
    realtime_start="1776-07-04",  # earliest possible
    realtime_end="9999-12-31",    # latest possible
    observation_start="2024-01-01",
)
payems = pl.DataFrame(vintages["observations"]).with_columns(
    pl.col("date", "realtime_start", "realtime_end").str.to_date(),
    pl.col("value").cast(pl.Float64, strict=False),
)
# Every published version of the June 2024 figure (thousands of jobs)
payems.filter(pl.col("date") == date(2024, 6, 1)).select("realtime_start", "realtime_end", "value")

To rebuild the series exactly as it looked on a given day, keep the versions whose window contains that day. On 1 August 2024, the latest month available was June, in its first estimate:

In [ ]:
def as_published_on(vintage_table: pl.DataFrame, day: date) -> pl.DataFrame:
    """The series exactly as FRED showed it on `day`."""
    live = (pl.col("realtime_start") <= day) & (pl.col("realtime_end") >= day)
    return vintage_table.filter(live).select("date", "value").sort("date")


as_published_on(payems, date(2024, 8, 1)).tail(3)

### Limits and gotchas

- Always send `file_type=json`: the default is XML.
- At most 100,000 rows per request: page with `offset` (long vintage histories can exceed it).
- FRED doesn't publish a rate limit; client libraries observe about 120 requests per minute, with `429` above it.
- Some series belong to third parties and carry their own terms. `SP500`, for example, has only the last 10 years on FRED. Read the series notes before republishing data.

**In your project:** `from finance_data import fred_series, alfred_vintages, alfred_as_of`

## 3. Tiingo: daily stock prices, raw and adjusted

**What you get:** end-of-day prices for tens of thousands of US stocks, ETFs and mutual funds, with 30+ years of history. Each row has raw prices (as traded), adjusted prices (split- and dividend-adjusted, CRSP method), and the dividend and split paid that day.

**Access:** sign up at [tiingo.com](https://www.tiingo.com), then *Account → API → Token*. Save it as `TIINGO_API_KEY`.

**Free plan:** 500 unique symbols per month, 50 requests per hour, 1,000 per day, 1 GB per month. The data is licensed for personal use only, so don't commit it to GitHub.

| Endpoint | Returns |
|---|---|
| `https://api.tiingo.com/tiingo/daily/<ticker>` | name, exchange, first and last price date |
| `https://api.tiingo.com/tiingo/daily/<ticker>/prices` | prices (only the latest day unless you pass `startDate`) |
| `https://apimedia.tiingo.com/docs/tiingo/daily/supported_tickers.zip` | every symbol Tiingo covers (no key needed) |

Price parameters: `startDate` and `endDate` (`YYYY-MM-DD`), `resampleFreq` (for example `weekly` or `monthly`), `format` (`json` or `csv`). The token goes in an `Authorization: Token <key>` header. A `token=` URL parameter also works, but the header keeps the key out of logs and error messages.

In [ ]:
TIINGO_HEADERS = {"Authorization": f"Token {require('TIINGO_API_KEY')}"}

resp = requests.get("https://api.tiingo.com/tiingo/daily/aapl", headers=TIINGO_HEADERS, timeout=30)
resp.raise_for_status()
meta = resp.json()
{key: meta[key] for key in ("ticker", "name", "exchangeCode", "startDate", "endDate")}

In [ ]:
resp = requests.get(
    "https://api.tiingo.com/tiingo/daily/aapl/prices",
    headers=TIINGO_HEADERS,
    params={"startDate": "2015-01-01", "endDate": "2025-12-31"},
    timeout=60,
)
resp.raise_for_status()
# [{"date": "2015-01-02T00:00:00.000Z", "open": ..., "close": ..., "volume": ...,
#   "adjOpen": ..., "adjClose": ..., "adjVolume": ..., "divCash": 0.0, "splitFactor": 1.0}, ...]

aapl = pl.DataFrame(resp.json(), infer_schema_length=None).with_columns(
    pl.col("date").str.slice(0, 10).str.to_date()
)
print(aapl.height, "days")
aapl.select("date", "close", "adjClose", "volume", "divCash", "splitFactor").tail()

### Raw versus adjusted prices

Apple split its shares 4-for-1 on 31 August 2020. The raw close drops by about three quarters overnight; the adjusted close doesn't. Use adjusted prices for returns, and raw prices for anything tied to the price people actually paid, such as a "price above $5" filter.

In [ ]:
aapl.filter(pl.col("date").is_between(date(2020, 8, 26), date(2020, 9, 2))).select(
    "date", "close", "adjClose", "splitFactor"
)

### Many tickers

At 50 requests per hour, the free plan fetches one ticker every 72 seconds, so a 500-stock universe takes about 10 hours. Save each ticker as soon as it arrives, so an error or a closed laptop never costs you the requests you already made:

In [ ]:
tickers_to_fetch = ["MSFT", "SPY"]

for ticker in tickers_to_fetch:
    path = DATA_DIR / f"tiingo_{ticker}.parquet"
    if path.exists():
        continue  # downloaded in an earlier run
    r = requests.get(
        f"https://api.tiingo.com/tiingo/daily/{ticker.lower()}/prices",
        headers=TIINGO_HEADERS,
        params={"startDate": "2015-01-01"},
        timeout=60,
    )
    r.raise_for_status()
    df = pl.DataFrame(r.json(), infer_schema_length=None).with_columns(
        pl.col("date").str.slice(0, 10).str.to_date(),
        pl.lit(ticker).alias("ticker"),
    )
    df.write_parquet(path)
    print(f"{ticker}: {df.height} rows saved")
    time.sleep(72)  # 50 requests per hour on the free plan

prices = pl.concat(
    [pl.read_parquet(DATA_DIR / f"tiingo_{t}.parquet") for t in tickers_to_fetch], how="vertical_relaxed"
)
prices.group_by("ticker").agg(
    pl.len().alias("days"), pl.col("date").min().alias("first"), pl.col("date").max().alias("last")
)

### The full symbol list, including stocks that stopped trading

`supported_tickers.zip` lists every symbol with its exchange, asset type and first and last price date. Symbols whose `endDate` is in the past stopped trading: delisted, acquired or renamed. You need them to avoid **survivorship bias**: a backtest on today's listed stocks silently drops every company that failed.

In [ ]:
zip_bytes = requests.get("https://apimedia.tiingo.com/docs/tiingo/daily/supported_tickers.zip", timeout=60).content
with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    universe = pl.read_csv(z.read(z.namelist()[0]), infer_schema_length=0)  # every column as text first

universe = universe.with_columns(pl.col("startDate", "endDate").str.to_date(strict=False))
us_stocks = universe.filter(
    (pl.col("assetType") == "Stock") & (pl.col("priceCurrency") == "USD") & pl.col("endDate").is_not_null()
)
cutoff = date.today() - timedelta(days=30)
status = pl.when(pl.col("endDate") < cutoff).then(pl.lit("stopped trading")).otherwise(pl.lit("trading"))
us_stocks.group_by(status.alias("status")).len()

### Limits and gotchas

- Share classes use a dash: `BRK-B`, not `BRK.B`.
- `401`: wrong token. `404`: unknown ticker. `429`: hourly or daily limit reached (the hourly limit resets every hour, the daily one at midnight US Eastern time).
- Without `startDate` you get only the latest day.
- The data is licensed for personal use: commit your code, not the downloaded files.

**In your project:** `from finance_data import tiingo_daily, tiingo_ticker_list`

## 4. Alpaca: bars for many symbols at once, from one minute to one month

**What you get:** bars, trades and quotes for US stocks and ETFs from 2016 onward, with many symbols per request.

**Free plan:** 200 requests per minute. Live data comes only from the IEX exchange (about 2.5% of US volume), but the consolidated feed from every exchange (SIP) is free once it is at least 15 minutes old, which covers historical research.

**Access:** sign up at [alpaca.markets](https://alpaca.markets), open the paper-trading dashboard and generate an API key. The secret is shown only once. Save them as `ALPACA_API_KEY` and `ALPACA_SECRET_KEY`.

**The request:** `GET https://data.alpaca.markets/v2/stocks/bars` with the headers `APCA-API-KEY-ID` and `APCA-API-SECRET-KEY`.

| Parameter | Meaning | Example |
|---|---|---|
| `symbols` | comma-separated tickers | `AAPL,MSFT,SPY` |
| `timeframe` | `1Min`, `5Min`, `15Min`, `1Hour`, `1Day`, `1Week`, `1Month` | `1Day` |
| `start`, `end` | `YYYY-MM-DD`, or a UTC time such as `2025-06-02T13:30:00Z` | |
| `feed` | `sip` (every exchange) or `iex` | `sip` |
| `adjustment` | `raw`, `split`, `dividend` or `all` | `all` |
| `limit`, `page_token` | up to 10,000 bars per page; send `next_page_token` back to get the next page | |

The response looks like `{"bars": {"AAPL": [{"t": …, "o": …, "h": …, "l": …, "c": …, "v": …, "n": …, "vw": …}, …], …}, "next_page_token": …}`: time, open, high, low, close, volume, number of trades and volume-weighted average price.

In [ ]:
ALPACA_HEADERS = {
    "APCA-API-KEY-ID": require("ALPACA_API_KEY"),
    "APCA-API-SECRET-KEY": require("ALPACA_SECRET_KEY"),
}


def get_bars(params: dict) -> pl.DataFrame:
    """Fetch every page of /v2/stocks/bars and return one long table."""
    params = {**params, "limit": 10_000}
    rows = []
    while True:
        resp = requests.get("https://data.alpaca.markets/v2/stocks/bars",
                            headers=ALPACA_HEADERS, params=params, timeout=60)
        if resp.status_code != 200:
            raise RuntimeError(f"Alpaca {resp.status_code}: {resp.text[:300]}")
        page = resp.json()
        for symbol, bars in (page.get("bars") or {}).items():
            rows += [{"symbol": symbol, **bar} for bar in bars]
        if not page.get("next_page_token"):
            break
        params["page_token"] = page["next_page_token"]  # ask for the next page
    return (
        pl.DataFrame(rows, infer_schema_length=None)
        .rename({"t": "timestamp", "o": "open", "h": "high", "l": "low", "c": "close",
                 "v": "volume", "n": "trades", "vw": "vwap"})
        .with_columns(pl.col("timestamp").str.to_datetime(time_zone="UTC"))
    )


daily = get_bars({"symbols": "AAPL,MSFT,SPY", "timeframe": "1Day", "start": "2024-01-01",
                  "end": "2025-12-31", "feed": "sip", "adjustment": "all"})
# Daily bars are stamped at midnight New York time: convert before taking the date
daily = daily.with_columns(
    pl.col("timestamp").dt.convert_time_zone("America/New_York").dt.date().alias("date")
)
daily.group_by("symbol").agg(
    pl.len().alias("days"), pl.col("date").min().alias("first"), pl.col("date").max().alias("last")
)

### Intraday bars

Same endpoint, smaller `timeframe`. US regular trading runs from 9:30 to 16:00 New York time, which is 13:30–20:00 UTC in summer (EDT) and 14:30–21:00 UTC in winter (EST).

In [ ]:
spy_5m = get_bars({"symbols": "SPY", "timeframe": "5Min", "start": "2025-06-02T13:30:00Z",
                   "end": "2025-06-02T19:55:00Z", "feed": "sip"})
print(spy_5m.height, "five-minute bars")
spy_5m.head()

### 4b. The same request with Alpaca's Python SDK

`alpaca-py` builds the request, follows the pages and returns pandas. Useful once you know what it does underneath.

In [ ]:
from alpaca.data.enums import Adjustment, DataFeed
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

client = StockHistoricalDataClient(require("ALPACA_API_KEY"), require("ALPACA_SECRET_KEY"))
request = StockBarsRequest(
    symbol_or_symbols=["AAPL", "MSFT"],
    timeframe=TimeFrame.Day,
    start=datetime(2024, 1, 1),
    end=datetime(2025, 12, 31),
    feed=DataFeed.SIP,
    adjustment=Adjustment.ALL,
)
sdk_bars = client.get_stock_bars(request).df  # pandas, indexed by (symbol, timestamp)
pl.from_pandas(sdk_bars.reset_index()).head()

### Limits and gotchas

- `403` mentioning recent SIP data: on the free plan, SIP data must end at least 15 minutes ago. Use an earlier `end`, or `feed=iex` for live data.
- Timestamps are UTC. Convert to `America/New_York` before grouping by trading day.
- History starts in 2016.
- If you lose the secret, generate a new key pair.

**In your project:** `from finance_data import alpaca_bars`

## 5. Alpha Vantage: fundamentals, earnings dates and delisted symbols

**What you get:** more than 100 endpoints: company fundamentals (income statement, balance sheet, cash flow, overview), quarterly earnings with announcement dates and estimates, lists of active and delisted symbols, FX, crypto, commodities and economic indicators.

**Access:** claim a key at [alphavantage.co/support/#api-key](https://www.alphavantage.co/support/#api-key) and save it as `ALPHAVANTAGE_API_KEY`.

**Free plan:** 25 requests per day, and this section uses 5 of them. Alpha Vantage offers unlimited access to verified open-source or educational projects, so it's worth asking their support.

| Free | Premium only |
|---|---|
| `TIME_SERIES_DAILY` with `outputsize=compact` (latest 100 days) | `TIME_SERIES_DAILY` with `outputsize=full` |
| `TIME_SERIES_WEEKLY`, `TIME_SERIES_MONTHLY` and their `_ADJUSTED` versions (full history) | `TIME_SERIES_DAILY_ADJUSTED` |
| `OVERVIEW`, `INCOME_STATEMENT`, `BALANCE_SHEET`, `CASH_FLOW`, `EARNINGS` | `TIME_SERIES_INTRADAY` |
| `LISTING_STATUS` (active or delisted symbols) | `INDEX_DATA` (S&P 500, VIX, …) |

**The request:** everything goes to `https://www.alphavantage.co/query` with `function`, `symbol` and `apikey`, plus `datatype=csv` if you want CSV instead of JSON.

**Errors arrive with status 200.** A bad symbol, the daily limit or a premium-only endpoint returns a normal-looking response whose JSON holds `Error Message`, `Information` or `Note` instead of data, so always check the body.

In [ ]:
AV_KEY = require("ALPHAVANTAGE_API_KEY")


def av_query(**params) -> requests.Response:
    """Call Alpha Vantage and raise if the body is an error message."""
    params["apikey"] = AV_KEY
    resp = requests.get("https://www.alphavantage.co/query", params=params, timeout=60)
    resp.raise_for_status()
    if resp.text.lstrip().startswith("{"):
        body = resp.json()
        for field in ("Error Message", "Information", "Note"):
            if field in body:
                raise RuntimeError(f"Alpha Vantage: {body[field]}")
    time.sleep(12)  # space calls out; bursts get throttled
    return resp

### Daily prices (the latest 100 days on the free plan)

Alpha Vantage's classic nested format: dates as keys, field names like `"1. open"`, and every number as a string.

In [ ]:
daily_json = av_query(function="TIME_SERIES_DAILY", symbol="IBM").json()
bars_by_day = daily_json["Time Series (Daily)"]  # {"2025-12-31": {"1. open": "...", ..., "5. volume": "..."}, ...}

ibm = pl.DataFrame([
    {"date": day, **{key.split(". ")[1]: float(value) for key, value in bar.items()}}
    for day, bar in bars_by_day.items()
]).with_columns(pl.col("date").str.to_date()).sort("date")
print(ibm.height, "days")
ibm.tail()

### Earnings dates and surprises

`reportedDate` is the day results were announced: the event date for an earnings study. `reportTime` says whether it came before the open or after the close.

In [ ]:
earnings = av_query(function="EARNINGS", symbol="IBM").json()
eps_columns = ["reportedEPS", "estimatedEPS", "surprise", "surprisePercentage"]
eps = pl.DataFrame(earnings["quarterlyEarnings"], infer_schema_length=None).with_columns(
    pl.col("fiscalDateEnding", "reportedDate").str.to_date(),
    pl.col(eps_columns).cast(pl.Float64, strict=False),  # the string "None" -> null
)
eps.head()

### Financial statements

`annualReports` and `quarterlyReports`, one record per period, every value a string. They carry the period end (`fiscalDateEnding`) but not the date the numbers became public: for point-in-time work, join them to `reportedDate` above or to SEC filing dates.

In [ ]:
income = av_query(function="INCOME_STATEMENT", symbol="IBM").json()
quarterly = pl.DataFrame(income["quarterlyReports"], infer_schema_length=None)
number_columns = [c for c in quarterly.columns if c not in ("fiscalDateEnding", "reportedCurrency")]
quarterly = quarterly.with_columns(
    pl.col("fiscalDateEnding").str.to_date(),
    pl.col(number_columns).cast(pl.Float64, strict=False),
)
quarterly.select("fiscalDateEnding", "totalRevenue", "grossProfit", "operatingIncome", "netIncome").head()

### Monthly adjusted prices as CSV (full history, free)

Add `datatype=csv` and read the body straight into Polars:

In [ ]:
monthly_csv = av_query(function="TIME_SERIES_MONTHLY_ADJUSTED", symbol="IBM", datatype="csv")
monthly = pl.read_csv(io.BytesIO(monthly_csv.content), try_parse_dates=True).sort("timestamp")
monthly.tail()  # timestamp, open, high, low, close, adjusted close, volume, dividend amount

### Delisted symbols

`LISTING_STATUS` returns every delisted symbol (or, with `state=active`, every active one) as CSV: a free way to build a universe without survivorship bias. Add `date=YYYY-MM-DD` to get the list as it stood on a past date.

In [ ]:
delisted_csv = av_query(function="LISTING_STATUS", state="delisted")
delisted = pl.read_csv(io.BytesIO(delisted_csv.content), infer_schema_length=0).with_columns(
    pl.col("ipoDate", "delistingDate").str.to_date(strict=False)
)
print(f"{delisted.height:,} delisted symbols")
delisted.sort("delistingDate", descending=True).head()

### Limits and gotchas

- Check the body, not only the status code: `av_query` above does it for you.
- 25 calls a day go fast. Cache every response, and test loops on one symbol first.
- Numbers are strings and missing values are the string `"None"`: cast with `strict=False`.

**In your project:** `from finance_data import av_statement, av_earnings, av_monthly_adjusted`

## 6. Databento: exchange-grade data, priced per request

**What you get:** historical and live data straight from exchange feeds: every order (`mbo`), order-book levels (`mbp-1`, `mbp-10`), trades, and bars from one second to one day, for US stocks, options (OPRA) and futures (CME). This matters if your project is about market microstructure or intraday trading.

**Cost:** usage-based, per byte of data. New accounts get **$125 of credit** for historical data, valid for six months. A payment card is required at sign-up but is only charged beyond the credit.

**Access:** sign up at [databento.com](https://databento.com), create an API key and save it as `DATABENTO_API_KEY`. Use the official client (`uv add databento`) rather than raw HTTP: the data comes in Databento's binary DBN format, which the client decodes.

| Term | Meaning | Examples |
|---|---|---|
| dataset | which venue or feed | `EQUS.SUMMARY` (US equities, daily summary), `XNAS.ITCH` (Nasdaq), `OPRA.PILLAR` (US options), `GLBX.MDP3` (CME futures) |
| schema | what kind of record | `ohlcv-1d`, `ohlcv-1m`, `trades`, `mbp-1`, `mbo` |
| symbols | up to 2,000 per request | `["AAPL", "MSFT"]` |
| start, end | time range; `end` is excluded | `"2025-01-01"`, `"2026-01-01"` |

In [ ]:
import databento as db

client = db.Historical(require("DATABENTO_API_KEY"))

# Metadata calls: what exists, and for which dates
print(client.metadata.list_datasets())
print(client.metadata.list_schemas(dataset="EQUS.SUMMARY"))
print(client.metadata.get_dataset_range(dataset="EQUS.SUMMARY"))

**Always price a request before you run it.** `get_cost` takes the same arguments as the download and returns the price in dollars:

In [ ]:
query = {
    "dataset": "EQUS.SUMMARY",
    "symbols": ["AAPL", "MSFT", "NVDA"],
    "schema": "ohlcv-1d",
    "start": "2025-01-01",
    "end": "2026-01-01",  # excluded
}
cost = client.metadata.get_cost(**query)
print(f"This request would cost ${cost:.4f} of your credit")

In [ ]:
MAX_COST = 0.50  # the most you're willing to spend on this cell

if cost > MAX_COST:
    raise RuntimeError(f"${cost:.2f} is above MAX_COST (${MAX_COST:.2f}): narrow the request first")

store = client.timeseries.get_range(**query)             # a DBNStore holding the downloaded records
store.to_file(DATA_DIR / "databento_daily.dbn.zst")      # keep the raw file: reloading it is free
daily_db = pl.from_pandas(store.to_df().reset_index())   # prices as floats, ts_event in UTC
daily_db.select("ts_event", "symbol", "open", "high", "low", "close", "volume").head()

### Limits and gotchas

- Call `get_cost` before every new kind of request. `symbols="ALL_SYMBOLS"` or tick-level schemas over long periods can cost more than the whole credit.
- `get_range` holds the whole result in memory. For large downloads use batch jobs (`client.batch.submit_job(...)`), which deliver files.
- Reload a saved file with `db.DBNStore.from_file(path)` instead of downloading it again.

**In your project:** `from finance_data import databento_ohlcv` (it refuses requests above `max_cost_usd`)

## 7. Stooq: free daily prices for global markets

**What you get:** daily, weekly and monthly prices for stocks, ETFs, indices, currencies and crypto from many countries, with long histories, as CSV.

**Access (changed in 2026):** Stooq now requires an `apikey` for CSV downloads; scripts that worked for years get a text page instead of data. To get a key, open https://stooq.com/q/d/?s=aapl.us&get_apikey, type the captcha, and copy the `apikey=` value from the CSV download link at the bottom of the page. Save it as `STOOQ_API_KEY`.

**The request:** `GET https://stooq.com/q/d/l/`

| Parameter | Meaning | Example |
|---|---|---|
| `s` | symbol | `aapl.us`, `^spx`, `eurusd`, `btcusd` |
| `i` | interval: `d`, `w`, `m`, `q`, `y` | `d` |
| `d1`, `d2` | date range as `YYYYMMDD` | `20000101` |
| `apikey` | your key | |

In [ ]:
params = {"s": "^spx", "i": "d", "d1": "20000101", "d2": "20251231", "apikey": require("STOOQ_API_KEY")}
resp = requests.get("https://stooq.com/q/d/l/", params=params, timeout=60)
resp.raise_for_status()

# Stooq answers 200 even when something is wrong, so check that the body really is CSV
if not resp.text.startswith("Date,"):
    raise RuntimeError(f"Stooq sent a message instead of CSV: {resp.text[:150]!r}")

spx = pl.read_csv(io.BytesIO(resp.content), try_parse_dates=True)
print(spx.height, "days")
spx.tail()

### Limits and gotchas

- Problems come back as text with status `200`: a missing or expired key, an unknown symbol, or the daily download cap ("Exceeded the daily hits limit").
- Before computing returns, check how the prices are adjusted by looking at a known split, such as `aapl.us` around 31 August 2020.
- It's a small site: download each symbol once and cache it.

**In your project:** `from finance_data import stooq_daily`

## 8. yfinance: quick prototypes only

**What it is:** a popular open-source library that reads Yahoo Finance's website. No key. It is not an official API: it breaks when Yahoo changes its site, heavy use triggers `YFRateLimitError`, and Yahoo's terms allow personal use only. Use it for a first look at an idea, and one of the sources above for the data your results depend on.

In [ ]:
import yfinance as yf

wide = yf.download(["AAPL", "MSFT"], start="2024-01-01", end="2026-01-01", auto_adjust=True, progress=False)
wide.head()  # pandas; columns are (Price, Ticker) pairs such as ("Close", "AAPL")

In [ ]:
# Wide -> long: one row per (date, ticker), as a Polars DataFrame
frames = []
for ticker in wide.columns.get_level_values("Ticker").unique():
    one = wide.xs(ticker, axis=1, level="Ticker").reset_index()
    frames.append(pl.from_pandas(one).with_columns(pl.lit(ticker).alias("ticker")))

yf_prices = pl.concat(frames).rename(str.lower).with_columns(pl.col("date").cast(pl.Date))
yf_prices.head()

In [ ]:
apple = yf.Ticker("AAPL")
print(apple.splits.tail())     # split ratios by date
print(apple.dividends.tail())  # cash dividends by ex-date

### Limits and gotchas

- `auto_adjust=True` (the default) replaces `Close` with the adjusted close.
- Intraday history is short: 1-minute bars go back 30 days (and one request covers at most about a week), 2- to 90-minute bars 60 days, hourly bars about two years.
- On `YFRateLimitError`, wait and try again later, and cache everything you download.

**In your project:** `from finance_data import yf_daily`

## Appendix A. Bulk downloads: no API, still Python

Some of the best free datasets are files rather than APIs. Python downloads them the same way; the work is in the parsing.

### A1. SEC insider transactions (Forms 3, 4 and 5)

One ZIP per quarter since 2006, holding tab-separated tables linked by `ACCESSION_NUMBER`: `SUBMISSION` (filing date, company, ticker), `REPORTINGOWNER` (who traded: officer, director, 10% owner), `NONDERIV_TRANS` (the stock transactions), plus derivative tables, holdings and footnotes. Dates look like `05-JAN-2026`. Transaction code `P` is an open-market or private purchase, `S` a sale.

Copy the links from the [SEC page](https://www.sec.gov/data-research/sec-markets-data/insider-transactions-data-sets): the newest quarter sits in a different folder from the older ones. The SEC `User-Agent` rule applies to these files too.

In [ ]:
url = "https://www.sec.gov/files/structureddata/data/insider-transactions-data-sets/2026q1_form345.zip"
resp = requests.get(url, headers={"User-Agent": require("SEC_USER_AGENT")}, timeout=120)
resp.raise_for_status()
insider_zip = zipfile.ZipFile(io.BytesIO(resp.content))
print(insider_zip.namelist())


def read_table(name: str) -> pl.DataFrame:
    """Read one tab-separated table from the ZIP, every column as text."""
    member = next(n for n in insider_zip.namelist() if Path(n).stem.upper() == name)
    return pl.read_csv(insider_zip.read(member), separator="\t", quote_char=None, infer_schema_length=0)


transactions = read_table("NONDERIV_TRANS")
submissions = read_table("SUBMISSION")
purchases = (
    transactions.filter(pl.col("TRANS_CODE") == "P")  # P = open-market or private purchase
    .join(submissions.select("ACCESSION_NUMBER", "FILING_DATE", "ISSUERTRADINGSYMBOL"), on="ACCESSION_NUMBER")
    .with_columns(
        pl.col("TRANS_DATE", "FILING_DATE").str.to_date("%d-%b-%Y", strict=False),  # 05-JAN-2026
        pl.col("TRANS_SHARES", "TRANS_PRICEPERSHARE").cast(pl.Float64, strict=False),
    )
    .with_columns((pl.col("TRANS_SHARES") * pl.col("TRANS_PRICEPERSHARE")).alias("value_usd"))
)
print(f"{purchases.height:,} purchases")
purchases.select(
    "ISSUERTRADINGSYMBOL", "TRANS_DATE", "FILING_DATE", "TRANS_SHARES", "TRANS_PRICEPERSHARE", "value_usd"
).head()

Join `REPORTINGOWNER` on `ACCESSION_NUMBER` the same way to see who traded. `finance_data.sec_insider_trades(year, quarter)` does all the joins and handles both download folders.

### A2. Fama-French factors (Kenneth French Data Library)

Zipped CSV files: a few description lines, then the monthly table, then an annual table. Values are in percent, and `-99.99` or `-999` means missing. File names: `F-F_Research_Data_Factors` (3 factors, monthly), `F-F_Research_Data_Factors_daily`, `F-F_Research_Data_5_Factors_2x3` (5 factors), `F-F_Momentum_Factor`. The library rebuilds its full history every month, so old values can shift slightly: keep the file you used.

In [ ]:
url = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_Factors_CSV.zip"
french_zip = zipfile.ZipFile(io.BytesIO(requests.get(url, timeout=60).content))
text = french_zip.read(french_zip.namelist()[0]).decode("latin-1")
print(text[:300])

In [ ]:
lines = text.splitlines()
header_at = next(i for i, line in enumerate(lines) if line.lstrip().startswith(","))  # ",Mkt-RF,SMB,HML,RF"
factor_names = [c.strip() for c in lines[header_at].split(",")[1:] if c.strip()]

rows = []
for line in lines[header_at + 1:]:
    parts = [p.strip() for p in line.split(",")]
    if not parts[0].isdigit():  # the blank line before the annual table ends the monthly block
        break
    rows.append([parts[0], *map(float, parts[1:len(factor_names) + 1])])

ff3 = (
    pl.DataFrame(rows, schema=["yyyymm", *factor_names], orient="row")
    .with_columns(
        pl.date(pl.col("yyyymm").str.slice(0, 4).cast(pl.Int32), pl.col("yyyymm").str.slice(4, 2).cast(pl.Int32), 1)
        .dt.month_end()
        .alias("date"),
        *[pl.when(pl.col(f) > -99.99).then(pl.col(f) / 100).alias(f) for f in factor_names],  # % -> decimal
    )
    .select("date", *factor_names)
)
ff3.tail()

### A3. Freddie Mac single-family loan data

No API: register on Freddie Mac's Clarity site, then download the files by hand. Start with the sample files; the full dataset runs to tens of gigabytes. The files are pipe-delimited with no header row, and the column names are listed in the dataset's user guide:

```python
origination = pl.read_csv("data/raw/sample_orig_2020.txt", separator="|", has_header=False,
                          new_columns=[...])  # names from the user guide
```

## Appendix B. When a request fails

| What you see | Likely cause | Fix |
|---|---|---|
| SEC `403` | missing or generic `User-Agent`, or too fast | set `SEC_USER_AGENT` to your name and email; stay under 10 requests per second |
| SEC `404` | CIK not padded, or concept not used | `f"{cik:010d}"`; list concepts with `companyfacts` |
| FRED returns XML | `file_type` not set | add `file_type=json` |
| FRED `400` | wrong key or series ID | print `resp.text` |
| `401` or `403` from Tiingo or Alpaca | wrong key | re-copy the key into `.env`, re-run the setup cell and the section's first cell |
| Alpaca `403` on recent data | free plan, SIP data inside the last 15 minutes | earlier `end`, or `feed=iex` |
| Alpha Vantage `Information` message | daily limit or premium-only endpoint | wait until tomorrow, or use a free endpoint |
| Stooq text instead of CSV | missing key or daily cap | get a key, or wait a day |
| `429` from any API | too many requests | slow down, cache, retry later |
| `YFRateLimitError` | Yahoo throttling | wait, and cache what you download |

**Download once, reuse forever.** Wrap every fetch like this so re-running a notebook never repeats a request (the functions in `finance_data.py` already do):

```python
def cached(path, fetch):
    if path.exists():
        return pl.read_parquet(path)
    df = fetch()
    df.write_parquet(path)
    return df

dgs10 = cached(DATA_DIR / "fred_DGS10.parquet", lambda: fred_column("DGS10"))
```